# Invio multiplo con tracciamento di ricezione e bounce

Questo notebook mostra come inviare email personalizzate a una lista di destinatari e tracciare l'esito:

- **Bounce SMTP**: il messaggio è rifiutato immediatamente dal server (indirizzo inesistente, casella piena, …)
- **Ricezione confermata**: il destinatario risponde con la parola `RICEVUTO` nel testo, oppure il suo client invia una ricevuta automatica (MDN)

L'idea centrale è usare il `Message-ID` come **token di correlazione**: ogni destinatario riceve un messaggio con un ID univoco, conservato in un database locale; le risposte vengono abbinate all'ID per aggiornare il record corrispondente.

## Setup

Il database di tracciamento ha un record per ogni destinatario. La colonna `id` contiene il `Message-ID` che verrà usato nell'invio — è anche la chiave primaria, il che rende la correlazione con le risposte un semplice lookup per uguaglianza.

`make_msgid(domain='mergecontrol.example')` genera un ID conforme a RFC 5322 con un dominio sentinella non reale, scelto appositamente per filtrare in ricezione solo le risposte a questa campagna ignorando tutto il resto della casella.

Gli eventi rilevanti (invii, bounce, ricevute) vengono registrati tramite `logging`, configurato con un formato ispirato all'Apache Common Log Format.

In [1]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)-8s %(message)s',
    datefmt='%d/%b/%Y:%H:%M:%S %z',
)

In [2]:
from email.utils import make_msgid
import sqlite3

conn = sqlite3.connect(':memory:')

with conn:
    conn.executescript('''
    CREATE TABLE IF NOT EXISTS destinatari (
        id         TEXT PRIMARY KEY,
        email      TEXT NOT NULL,
        ricevuto   BOOLEAN,
        rimbalzato BOOLEAN
    );
    DELETE FROM destinatari;
    ''')

with conn:
    conn.executemany('INSERT INTO destinatari (id, email) VALUES (?, ?)', [
        (make_msgid(domain='mergecontrol.example'), 'santini@di.unimi.it'),
        (make_msgid(domain='mergecontrol.example'), 'santini@di.unimi.it'),
        (make_msgid(domain='mergecontrol.example'), 'nonesiste1@unimi.it'),
        (make_msgid(domain='mergecontrol.example'), 'nonesiste2@unimi.it'),
    ])

## Credenziali

Le credenziali vengono lette dal file `.env` nella directory corrente, che deve contenere le variabili `EMAIL_USER` (indirizzo completo, usato anche come mittente) e `EMAIL_PASSWORD`.

In [3]:
from pathlib import Path

env = dict(
    line.split('=', 1)
    for line in Path('.env').read_text().splitlines()
    if '=' in line and not line.startswith('#')
)

email_user     = env['EMAIL_USER']
email_password = env['EMAIL_PASSWORD']

## Invio

Ogni messaggio viene inviato singolarmente, in modo da poter personalizzare il testo (es. includere il nome del destinatario) e gestire gli errori in modo indipendente.

Il `Message-ID` assegnato è lo stesso ID del record nel database: quando arriverà una risposta, l'`In-Reply-To` header conterrà questo valore e il lookup sarà diretto.

Aggiungiamo `Disposition-Notification-To` per richiedere una ricevuta di lettura automatica (MDN, vedi sezione successiva). Non tutti i client la supportano: Outlook mostra una finestra di dialogo all'utente che può rifiutare, e molti client la ignorano del tutto — ma quando arriva non richiede azione manuale da parte del destinatario.

I **bounce SMTP immediati** (`SMTPRecipientsRefused`) vengono catturati messaggio per messaggio e aggiornano subito il database, senza attendere la fase di ricezione.

In [4]:
from email.message import EmailMessage
from email.utils import formatdate
from smtplib import SMTP, SMTPRecipientsRefused

with SMTP('smtp.di.unimi.it') as smtp:
    smtp.login(email_user, email_password)

    for id_, email in conn.execute('SELECT id, email FROM destinatari'):
        msg = EmailMessage()
        msg['From']    = email_user
        msg['To']      = email
        msg['Subject'] = 'Invio con controllo di ricezione'
        msg['Message-ID'] = id_
        msg['Date']    = formatdate()
        msg['Disposition-Notification-To'] = email_user
        msg.set_content(
            f'Testo del messaggio per {email}.\n\n'
            'Se hai ricevuto questo messaggio, rispondi includendo '
            'la parola RICEVUTO nel testo.'
        )
        try:
            smtp.send_message(msg)
            logging.info('Inviato   %s  %s', email, id_)
        except SMTPRecipientsRefused as e:
            logging.warning('Rifiutato %s — %s', email, e.recipients)
            with conn:
                conn.execute('UPDATE destinatari SET rimbalzato = 1 WHERE id = ?', (id_,))

[05/Jun/2026:14:54:50 +0200] INFO     Inviato   santini@di.unimi.it  <178066408940.321663.17899520438364122917@mergecontrol.example>
[05/Jun/2026:14:54:50 +0200] INFO     Inviato   santini@di.unimi.it  <178066408940.321663.17227934296963413257@mergecontrol.example>
[05/Jun/2026:14:54:50 +0200] INFO     Inviato   nonesiste1@unimi.it  <178066408940.321663.15777886076502938380@mergecontrol.example>
[05/Jun/2026:14:54:51 +0200] INFO     Inviato   nonesiste2@unimi.it  <178066408940.321663.4082539019356429816@mergecontrol.example>


## Standard RFC: DSN e MDN

Gli standard definiscono due meccanismi formali per notificare l'esito di una consegna.

### DSN — Delivery Status Notification (RFC 3464)

Un bounce "vero" è un messaggio `multipart/report; report-type=delivery-status` che il server destinatario invia all'**envelope sender** (l'indirizzo del `MAIL FROM`, non del `From:` header). La parte `message/delivery-status` contiene il campo `Original-Message-Id:` con il Message-ID originale:

```
Content-Type: multipart/report; report-type=delivery-status

--boundary
Content-Type: message/delivery-status

Original-Message-Id: <abc123@mergecontrol.example>
Status: 5.1.1
Action: failed
```

Per ricevere DSN occorre usare l'estensione SMTP `DSN` (`RCPT TO:<addr> NOTIFY=FAILURE`), supportata da molti MTA ma non esposta direttamente da `smtplib`.

### MDN — Message Disposition Notification (RFC 8098)

Le ricevute di lettura automatiche sono messaggi `multipart/report; report-type=disposition-notification`. A differenza dei DSN, **impostano `In-Reply-To`** al Message-ID originale — ciò le rende correlabili con lo stesso meccanismo usato per le risposte umane.

### Perché l'approccio standard non basta in pratica

In ambienti con Outlook e MTA aziendali il supporto è parziale:

- Outlook genera MDN solo su conferma esplicita dell'utente (o non li genera affatto)
- I bounce possono arrivare come normali email di testo, senza rispettare il formato `multipart/report`
- Molti MTA aziendali non abilitano `NOTIFY=SUCCESS`

Per questo motivo chiediamo esplicitamente al destinatario di rispondere con la parola `RICEVUTO` nel testo — un meccanismo che funziona indipendentemente da client e server. Il codice di ricezione gestisce comunque anche DSN e MDN quando presenti.

## Lettura delle risposte

Leggiamo la casella in arrivo e aggiorniamo il database. Ogni messaggio che ha `In-Reply-To` con il dominio sentinella viene classificato in tre categorie:

In [9]:
from imap_tools import MailBox

with MailBox('imap.di.unimi.it').login(email_user, email_password, 'INBOX') as mailbox:
    for msg in mailbox.fetch():
        in_reply_to_vals = msg.headers.get('in-reply-to', [])
        in_reply_to = in_reply_to_vals[0] if in_reply_to_vals else None
        if not in_reply_to or '@mergecontrol.example' not in in_reply_to:
            continue

        mid = in_reply_to.strip()
        report_type = msg.obj.get_param('report-type') or ''

        if 'delivery-status' in report_type:
            with conn:
                conn.execute('UPDATE destinatari SET rimbalzato = 1 WHERE id = ?', (mid,))
            logging.warning('Bounce (DSN)     %s', mid)
        elif 'disposition-notification' in report_type:
            with conn:
                conn.execute('UPDATE destinatari SET ricevuto = 1 WHERE id = ?', (mid,))
            logging.info('Ricevuta (MDN)   %s', mid)
        elif 'RICEVUTO' in (msg.text or '').upper():
            with conn:
                conn.execute('UPDATE destinatari SET ricevuto = 1 WHERE id = ?', (mid,))
            logging.info('Risposta umana   %s', mid)
        else:
            logging.warning('Non classificato %r da %s → %s', msg.subject, msg.from_, mid)

[05/Jun/2026:14:55:49 +0200] WARNING  Bounce (DSN)     <178066408940.321663.15777886076502938380@mergecontrol.example>
[05/Jun/2026:14:55:49 +0200] WARNING  Bounce (DSN)     <178066408940.321663.4082539019356429816@mergecontrol.example>
[05/Jun/2026:14:55:49 +0200] INFO     Risposta umana   <178066408940.321663.17227934296963413257@mergecontrol.example>


In [10]:
with conn:
  for row in conn.execute('SELECT * FROM destinatari'):
    print(row)

('<178066408940.321663.17899520438364122917@mergecontrol.example>', 'santini@di.unimi.it', None, None)
('<178066408940.321663.17227934296963413257@mergecontrol.example>', 'santini@di.unimi.it', 1, None)
('<178066408940.321663.15777886076502938380@mergecontrol.example>', 'nonesiste1@unimi.it', None, 1)
('<178066408940.321663.4082539019356429816@mergecontrol.example>', 'nonesiste2@unimi.it', None, 1)
